In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
from tqdm import tqdm

In [8]:
def extract_pose_from_video(video_path, output_csv):
    mp_pose = mp.solutions.pose

    pose = mp_pose.Pose(
        static_image_mode=False, # for video processing
        model_complexity=1, # to reduce computation, higher means more accuracy
        enable_segmentation=False, # whether we want to separate the shilloute of the person
        min_detection_confidence=0.5, #the threshold which decides whether mediapipe classifies the frame as having a human or not
        min_tracking_confidence=0.5 # confidence for pose tracking, if the confidence is below thresh, mp runs the pose detection again
    )
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Check the path video not loaded")
        return
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    data = []
    frame_id = 0

    print("Processing video...")
    with tqdm(total=frame_count) as pbar:

        while True:
            ret, frame = cap.read() # ret is true if frame exists, helps determine when vid ends

            if not ret:
                break

            # Convert BGR → RGB
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            results = pose.process(image) # run the nueral network to identify the pose

            if results.pose_landmarks: # mp estimates 33 human pose landmarks, stores each as dict in a list, else none

                for landmark_id, landmark in enumerate(results.pose_landmarks.landmark):

                    data.append({
                        "frame": frame_id,
                        "landmark": landmark_id,
                        "x": landmark.x,
                        "y": landmark.y, # normalized coordinates 0->1, x=0.5 means centre of frame
                        "z": landmark.z,
                        "visibility": landmark.visibility
                    })

            frame_id += 1
            pbar.update(1) #progress bar update for tqdm

    cap.release() #close vid file

    df = pd.DataFrame(data) 

    df.to_csv(output_csv, index=False)

    print(f"\nPose data saved to: {output_csv}")

In [ ]:
print("please work")

In [10]:
video_path = "../data/raw_videos/squat_test.mp4"
output_csv = "../outputs/keypoints/squat_keypoints.csv"

extract_pose_from_video(video_path, output_csv)

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [11]:
print(mp.__version__)

0.10.32


In [12]:
print(dir(mp))

['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [ ]:
print("test")